# Carpark Matching — How We Merge Everything

GoPark pulls carpark data from 6 different sources. None of them have everything — government data has locations but no rates, community sites have rates but different IDs, commercial sites have availability but no coordinates.

This notebook walks through how we merge them into one unified dataset. Each source is layered on top using a 3-tier matching strategy:

1. **Exact ID** — same `car_park_no` across sources
2. **Proximity** — within ~111 metres (0.001° lat/lng)
3. **Name similarity** — normalized string matching

The merge runs every 6 hours in the backend cron. Order matters — earlier sources set the baseline, later ones enrich.

In [ ]:
import os
import re
import json
import math
import requests
import pandas as pd
from pyproj import Transformer
from dotenv import load_dotenv

load_dotenv()

DATAMALL_API = os.environ["DATAMALL_API"]
DATAGOVSG_KEY = os.environ.get("DATAGOVSG_KEY", "")
transformer = Transformer.from_crs("EPSG:3414", "EPSG:4326", always_xy=True)

## The Matching Functions

Three ways to match an incoming carpark to one we already have. They're tried in order — exact ID first, then proximity, then name. First match wins.

The proximity threshold is `0.001` degrees (~111m at the equator). Name matching normalises both strings to lowercase alphanumeric, then checks if they're equal or one contains the other (minimum 4 chars to avoid false positives).

In [ ]:
PROXIMITY_THRESHOLD = 0.001

def normalize(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def find_by_proximity(merged, lat, lng):
    best, best_d = None, PROXIMITY_THRESHOLD
    for cp in merged.values():
        d = math.hypot(cp["lat"] - lat, cp["lng"] - lng)
        if d < best_d:
            best_d = d
            best = cp
    return best

def find_by_name(merged, name):
    norm = normalize(name)
    if len(norm) < 4:
        return None
    for cp in merged.values():
        existing = normalize(cp["name"])
        if not existing:
            continue
        if norm == existing or norm in existing or existing in norm:
            return cp
    return None

def find_existing(merged, cp_id, name, lat, lng):
    if cp_id in merged:
        return merged[cp_id]
    by_prox = find_by_proximity(merged, lat, lng)
    if by_prox:
        return by_prox
    return find_by_name(merged, name)

def add_source(cp, source, source_id):
    if source not in cp["sources"]:
        cp["sources"].append(source)
    cp["source_ids"][source] = source_id

print(f"Proximity threshold: {PROXIMITY_THRESHOLD}° ≈ {PROXIMITY_THRESHOLD * 111_000:.0f}m")
print(f"Name matching: lowercase alphanumeric, min 4 chars, substring match")

## Layer 1 — Government HDB (The Base)

Everything starts here. The data.gov.sg dataset gives us ~2265 HDB carparks with SVY21 coordinates (converted to WGS84), addresses, carpark type, and metadata like free parking periods.

These carpark IDs (`ACB`, `J14`, `NT9`) become the primary keys in our merged dataset. Every other source tries to match against these first.

In [ ]:
merged = {}

gov_url = "https://data.gov.sg/api/action/datastore_search"
params = {"resource_id": "d_23f946fa557947f93a8043bbef41dd09", "limit": 3000}
gov_data = requests.get(gov_url, params=params, timeout=30).json()
gov_df = pd.DataFrame(gov_data["result"]["records"])
gov_df["x_coord"] = pd.to_numeric(gov_df["x_coord"], errors="coerce")
gov_df["y_coord"] = pd.to_numeric(gov_df["y_coord"], errors="coerce")
gov_df = gov_df.dropna(subset=["x_coord", "y_coord"])
gov_df = gov_df[(gov_df["x_coord"] > 0) & (gov_df["y_coord"] > 0)]

lngs, lats = transformer.transform(gov_df["x_coord"].values, gov_df["y_coord"].values)
gov_df["lat"] = lats
gov_df["lng"] = lngs

for _, row in gov_df.iterrows():
    cp_id = row["car_park_no"]
    merged[cp_id] = {
        "id": cp_id,
        "name": row["address"],
        "lat": row["lat"],
        "lng": row["lng"],
        "agency": "HDB",
        "sources": ["gov"],
        "source_ids": {"gov": cp_id},
    }

print(f"Layer 1: {len(merged)} HDB carparks from data.gov.sg")
print(f"Sample: {list(merged.keys())[:5]}")

## Layer 2 — LTA DataMall

DataMall has ~2600 records across three agencies. The HDB records use the **same IDs** as data.gov.sg, so they match by exact ID — no fuzzy matching needed.

The interesting part: it adds **40 LTA** and **89 URA** carparks that aren't in the government dataset. These are commercial malls (Suntec City, VivoCity, ION Orchard) and private developments. We prefix their IDs to avoid collisions — `LTA-3` for Raffles City, `URA-A0007` for Angullia Park.

In [ ]:
dm_url = "https://datamall2.mytransport.sg/ltaodataservice/CarParkAvailabilityv2"
headers = {"AccountKey": DATAMALL_API}

all_dm = []
skip = 0
while True:
    batch = requests.get(dm_url, headers=headers, params={"$skip": skip}, timeout=30).json().get("value", [])
    if not batch:
        break
    all_dm.extend(batch)
    skip += 500

dm_df = pd.DataFrame(all_dm)

matched_hdb, new_lta, new_ura = 0, 0, 0
seen_dm = set()

for _, row in dm_df.iterrows():
    agency = row["Agency"]
    raw_id = str(row["CarParkID"]).strip()
    name = row.get("Development", "")
    loc = row.get("Location", "")

    if agency == "HDB":
        if raw_id in merged:
            add_source(merged[raw_id], "datamall", raw_id)
            matched_hdb += 1
        continue

    cp_id = f"{agency}-{raw_id}"
    if cp_id in seen_dm:
        continue
    seen_dm.add(cp_id)

    parts = loc.split()
    lat = float(parts[0]) if len(parts) == 2 else 0
    lng = float(parts[1]) if len(parts) == 2 else 0

    existing = find_existing(merged, cp_id, name, lat, lng)
    if existing:
        add_source(existing, "datamall", cp_id)
    else:
        merged[cp_id] = {
            "id": cp_id, "name": name, "lat": lat, "lng": lng,
            "agency": agency, "sources": ["datamall"], "source_ids": {"datamall": cp_id},
        }
        if agency == "LTA":
            new_lta += 1
        else:
            new_ura += 1

print(f"Layer 2: {len(dm_df)} DataMall records")
print(f"  HDB matched by exact ID: {matched_hdb}")
print(f"  New LTA carparks: {new_lta}")
print(f"  New URA carparks: {new_ura}")
print(f"  Running total: {len(merged)}")

## Layer 3 — Parkaholic

Parkaholic has ~4000 entries and uses the **same government carpark IDs** for HDB carparks. So those match by exact ID, just like DataMall.

For non-HDB carparks (commercial ones), Parkaholic uses its own sequential IDs (`1`, `2`, `3`...) which happen to overlap with DataMall's numbering for malls — `1` is Suntec City, `2` is Marina Square, `3` is Raffles City. We match these by proximity and name.

The main value Parkaholic adds: **parking rates** and **lot types** (Car, Motorcycle, Heavy Vehicle).

In [ ]:
park_data = requests.get("https://parkaholic.sg/data.json", timeout=30).json()

matched_id, matched_prox, matched_name, new_park = 0, 0, 0, 0

for cpid, info in park_data.items():
    name = info.get("Development", "").strip()
    parts = info.get("Location", "").split()
    if len(parts) != 2:
        continue
    try:
        lat, lng = float(parts[0]), float(parts[1])
    except ValueError:
        continue

    if cpid in merged:
        add_source(merged[cpid], "parkaholic", cpid)
        matched_id += 1
    else:
        by_prox = find_by_proximity(merged, lat, lng)
        if by_prox:
            add_source(by_prox, "parkaholic", cpid)
            matched_prox += 1
        else:
            by_name = find_by_name(merged, name)
            if by_name:
                add_source(by_name, "parkaholic", cpid)
                matched_name += 1
            else:
                merged[cpid] = {
                    "id": cpid, "name": name, "lat": lat, "lng": lng,
                    "agency": "", "sources": ["parkaholic"], "source_ids": {"parkaholic": cpid},
                }
                new_park += 1

print(f"Layer 3: {len(park_data)} Parkaholic entries")
print(f"  Matched by exact ID: {matched_id}")
print(f"  Matched by proximity: {matched_prox}")
print(f"  Matched by name: {matched_name}")
print(f"  New carparks: {new_park}")
print(f"  Running total: {len(merged)}")

## Layer 4 — SGCarMart

This is where matching gets interesting. SGCarMart uses completely sequential IDs (`1`, `2`, `3`...) with **no relation** to the government system. No shared IDs at all.

So we rely entirely on proximity and name matching. Each SGCarMart entry has coordinates and a name like "National Library Building" or "VivoCity" — we try to find these in our existing merged dataset.

We prefixed their IDs as `SGC-{id}` to avoid collisions.

In [ ]:
sgc_df = pd.read_csv("rates/rates.csv", index_col=0)

matched_prox, matched_name, new_sgc = 0, 0, 0
sgc_examples = {"proximity": [], "name": [], "new": []}

for _, row in sgc_df.iterrows():
    sgc_id = f"SGC-{row['id']}"
    name = str(row.get("name", ""))
    lat = row.get("latitude", 0)
    lng = row.get("longitude", 0)
    if not lat or not lng:
        continue

    by_prox = find_by_proximity(merged, lat, lng)
    if by_prox:
        add_source(by_prox, "sgcarmart", sgc_id)
        matched_prox += 1
        if len(sgc_examples["proximity"]) < 3:
            sgc_examples["proximity"].append(f"  {name} → {by_prox['name']}")
    else:
        by_name = find_by_name(merged, name)
        if by_name:
            add_source(by_name, "sgcarmart", sgc_id)
            matched_name += 1
            if len(sgc_examples["name"]) < 3:
                sgc_examples["name"].append(f"  {name} → {by_name['name']}")
        else:
            merged[sgc_id] = {
                "id": sgc_id, "name": name, "lat": lat, "lng": lng,
                "agency": "", "sources": ["sgcarmart"], "source_ids": {"sgcarmart": sgc_id},
            }
            new_sgc += 1
            if len(sgc_examples["new"]) < 3:
                sgc_examples["new"].append(f"  {name}")

print(f"Layer 4: {len(sgc_df)} SGCarMart entries")
print(f"  Matched by proximity: {matched_prox}")
print(f"  Matched by name: {matched_name}")
print(f"  New carparks: {new_sgc}")
print(f"\nProximity match examples:")
print("\n".join(sgc_examples["proximity"]))
print(f"\nName match examples:")
print("\n".join(sgc_examples["name"]))
print(f"\nNew (unmatched) examples:")
print("\n".join(sgc_examples["new"]))
print(f"\nRunning total: {len(merged)}")

## Layer 5 — CapitaLand JustPark

JustPark covers 91 CapitaLand properties — malls (Raffles City, Funan, IMM), offices (Capital Tower, CapitaSpring), and business parks.

JustPark doesn't provide coordinates, so we match **by name only**. Most of these malls already exist in our dataset from DataMall/Parkaholic. JustPark's main value: it's the only source with **total capacity** for CapitaLand carparks, plus detailed rate text.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})

page = session.get("https://justpark.capitaland.com/Lot-Availability", timeout=30)
token_match = re.search(r'name="__RequestVerificationToken"[^>]*value="([^"]*)"', page.text)
form_token = token_match.group(1)

api_headers = {
    "X-Requested-With": "XMLHttpRequest",
    "__RequestVerificationToken": form_token,
    "x-ModID": "FELotAvail",
    "Origin": "https://justpark.capitaland.com",
    "Referer": "https://justpark.capitaland.com/Lot-Availability",
    "X-APIAction": "SelectSiteNoAuth/OnSelectSite",
    "Content-Type": "multipart/form-data; boundary=----Boundary",
}

body = '------Boundary\r\nContent-Disposition: form-data; name="JData"\r\n\r\n{"DisplayType":"LotAvail","IsAuth":"false"}\r\n------Boundary--\r\n'
resp = session.post("https://justpark.capitaland.com/AjaxNoAuth/OnHttpPost", headers=api_headers, data=body, timeout=30)
sites = json.loads(resp.json()["Result"])

matched_jp, new_jp = 0, 0
jp_examples = []

for site in sites:
    name = site["SiteDesc"]
    jp_id = f"CLND-{site['SiteCode']}"

    existing = find_by_name(merged, name)
    if existing:
        add_source(existing, "justpark", jp_id)
        matched_jp += 1
        if len(jp_examples) < 5:
            jp_examples.append(f"  {name} → {existing['name']} ({existing['id']})")
    else:
        merged[jp_id] = {
            "id": jp_id, "name": name, "lat": 0, "lng": 0,
            "agency": "CapitaLand", "sources": ["justpark"], "source_ids": {"justpark": jp_id},
        }
        new_jp += 1

print(f"Layer 5: {len(sites)} JustPark sites")
print(f"  Matched by name: {matched_jp}")
print(f"  New (unmatched): {new_jp}")
print(f"\nMatch examples:")
print("\n".join(jp_examples))
print(f"\nRunning total: {len(merged)}")

## Layer 6 — URA Capacity (Enrichment Only)

This layer doesn't add new carparks — it fills in missing **total capacity** for URA carparks. DataMall only gives us available lots ("224 available"), but not how many lots there are total. The URA Parking Places dataset fills that gap with exact counts for cars, motorcycles, and heavy vehicles.

Matching is by ID — URA's `PP_CODE` directly matches DataMall's URA carpark IDs (both use `A0007`, `A0021`, etc).

In [ ]:
ura_meta = requests.get(
    "https://api-open.data.gov.sg/v1/public/api/datasets/d_9bf8620ecfdc8a5f8f77e3f02160af5c/poll-download",
    timeout=30,
).json()
ura_places = requests.get(ura_meta["data"]["url"], timeout=60).json()

ura_cap = {}
for f in ura_places["features"]:
    pp = f["properties"]["PP_CODE"]
    ura_cap[pp] = {
        "name": f["properties"]["PARKING_PL"],
        "cars": f["properties"].get("NO_CAR", 0) or 0,
        "motorcycles": f["properties"].get("NO_MCYCLE", 0) or 0,
        "heavy": f["properties"].get("NO_H_VEHIC", 0) or 0,
    }

enriched = 0
for cp in merged.values():
    dm_id = cp["source_ids"].get("datamall", "")
    ura_id = dm_id.replace("URA-", "") if dm_id.startswith("URA-") else None
    if ura_id and ura_id in ura_cap:
        enriched += 1

print(f"Layer 6: {len(ura_places['features'])} URA parking places")
print(f"  Enriched with capacity: {enriched}")
print(f"  Total cars capacity: {sum(p['cars'] for p in ura_cap.values()):,}")
print(f"  Total motorcycle capacity: {sum(p['motorcycles'] for p in ura_cap.values()):,}")
print(f"\nFinal total: {len(merged)} carparks")

## Geofence — Singapore Only

Parkaholic includes carparks in Johor Bahru and other Malaysian areas near the border. The final step filters everything through a Singapore boundary polygon (built from merged data.gov.sg coastline + OpenStreetMap data — see `singaporeBoundary.ipynb`).

In [ ]:
from shapely.geometry import shape, Point

with open("singapore.geojson") as f:
    sg_geojson = json.load(f)

border = shape(sg_geojson["features"][0]["geometry"])

before = len(merged)
excluded = []
for cp_id in list(merged.keys()):
    cp = merged[cp_id]
    if cp["lat"] == 0 and cp["lng"] == 0:
        continue
    if not border.contains(Point(cp["lng"], cp["lat"])):
        excluded.append(f"  {cp['name']} ({cp['lat']:.4f}, {cp['lng']:.4f})")
        del merged[cp_id]

print(f"Geofence: {len(excluded)} carparks outside Singapore removed")
print(f"  Before: {before} → After: {len(merged)}")
if excluded[:5]:
    print(f"\nExcluded (sample):")
    print("\n".join(excluded[:5]))

## Final Stats

Summary of the merged dataset — how many carparks, how many sources each has, and the source breakdown.

In [ ]:
source_counts = {}
multi_source = 0
for cp in merged.values():
    n = len(cp["sources"])
    if n > 1:
        multi_source += 1
    for s in cp["sources"]:
        source_counts[s] = source_counts.get(s, 0) + 1

by_num_sources = {}
for cp in merged.values():
    n = len(cp["sources"])
    by_num_sources[n] = by_num_sources.get(n, 0) + 1

print(f"=== Merged Dataset ===")
print(f"Total carparks: {len(merged)}")
print(f"Multi-source carparks: {multi_source} ({multi_source/len(merged)*100:.1f}%)")
print(f"\nCarparks per source:")
for src, count in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {src:<12s} {count:>5d}")
print(f"\nBy number of sources:")
for n in sorted(by_num_sources):
    print(f"  {n} source{'s' if n > 1 else ''}: {by_num_sources[n]:>5d}")

best = max(merged.values(), key=lambda cp: len(cp["sources"]))
print(f"\nMost-sourced carpark: {best['name']}")
print(f"  Sources: {', '.join(best['sources'])}")
print(f"  IDs: {best['source_ids']}")

## Merge Flow Summary

| Order | Source | ID System | Match Method | Adds |
|-------|--------|-----------|--------------|------|
| 1 | **data.gov.sg** | `ACB`, `J14` | — (base layer) | ~2265 HDB carparks, locations, metadata |
| 2 | **LTA DataMall** | `HDB-id`, `LTA-n`, `URA-code` | Exact ID (HDB), new (LTA/URA) | ~130 commercial carparks, live availability |
| 3 | **Parkaholic** | Same gov IDs + sequential | Exact ID → Proximity → Name | Rates, lot types, availability |
| 4 | **SGCarMart** | `SGC-n` (sequential) | Proximity → Name | Rates (weekday/sat/sun text) |
| 5 | **JustPark** | `CLND-code` | Name only | Total capacity, rates for CapitaLand |
| 6 | **URA Capacity** | `PP_CODE` = URA IDs | Exact ID | Total lot counts (cars, motorcycles, heavy) |
| ∞ | **Geofence** | — | Singapore boundary polygon | Removes ~260 JB/Malaysian entries |

The backend runs this merge every 6 hours via cron. Availability refreshes every 60 seconds from gov.sg and DataMall.